In [1]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from dataset_ood_download import get_data_list

/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_tag = "ablation-full-tasks_0509"

tasks = [
    "bace",
    "smol-property_prediction-bbbp",
    "smol-property_prediction-clintox",
    "smol-property_prediction-hiv",
    "smol-property_prediction-sider",
    "smol-property_prediction-esol",
    "smol-property_prediction-lipo",
    "qm9_homo",
    "qm9_lumo",
    "qm9_homo_lumo_gap",
    "forward_reaction_prediction",
    "retrosynthesis",
    "reagent_prediction",
    "chebi-20-text2mol",
    "chebi-20-mol2text",
]

path_template = "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{}_{}_0219"

In [3]:
trainset_name = f'/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_{data_tag}'
train_datasets = {}
for task in tasks:
    path = path_template.format("train", task)
    train_datasets[task] = datasets.load_from_disk(path)

# concat train_datasets
list_train_datasets = []
for task in tasks:
    list_train_datasets.append(train_datasets[task])

concat_train_datasets = datasets.concatenate_datasets(list_train_datasets)
concat_train_datasets.save_to_disk(trainset_name)



Saving the dataset (14/14 shards): 100%|██████████| 854234/854234 [01:29<00:00, 9583.47 examples/s] 


In [4]:
train_datasets

{'bace': Dataset({
     features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
     num_rows: 1210
 }),
 'smol-property_prediction-bbbp': Dataset({
     features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
     num_rows: 1569
 }),
 'smol-property_prediction-clintox': Dataset({
     features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
     num_rows: 1144
 }),
 'smol-property_prediction-hiv': Dataset({
     features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
     num_rows: 32864
 }),
 'smol-property_prediction-sider': Dataset({
     features

In [6]:
list(set(concat_train_datasets['task']))

['smol-property_prediction-lipo',
 'retrosynthesis',
 'chebi-20-text2mol',
 'smol-property_prediction-esol',
 'smol-property_prediction-bbbp',
 'reagent_prediction',
 'chebi-20-mol2text',
 'qm9_lumo',
 'smol-property_prediction-hiv',
 'smol-property_prediction-clintox',
 'qm9_homo_lumo_gap',
 'qm9_homo',
 'smol-property_prediction-sider',
 'bace',
 'forward_reaction_prediction']

In [7]:
testset_name = f'/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_{data_tag}'
test_datasets = {}
for task in tasks:
    path = path_template.format("test", task)
    test_datasets[task] = datasets.load_from_disk(path)

# concat test_datasets
list_test_datasets = []
for task in tasks:
    list_test_datasets.append(test_datasets[task])


ood_test_data_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_ood_0411'
ood_test_data = datasets.load_from_disk(ood_test_data_path)

list_test_datasets.append(ood_test_data)

concat_test_datasets = datasets.concatenate_datasets(list_test_datasets)
concat_test_datasets.save_to_disk(testset_name)

Saving the dataset (1/1 shards): 100%|██████████| 27508/27508 [00:03<00:00, 7425.48 examples/s] 


In [8]:
concat_test_datasets

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
    num_rows: 27508
})

In [9]:
list(set(concat_test_datasets['task']))

['smol-property_prediction-lipo',
 'alchemy_homo_lumo_gap',
 'smol-property_prediction-bbbp',
 'reagent_prediction',
 'smol-property_prediction-clintox',
 'qm9_homo',
 'bace',
 'aqsol-logS',
 'chebi-20-mol2text',
 'orderly-retrosynthesis',
 'retrosynthesis',
 'alchemy_homo',
 'qm9_lumo',
 'smol-property_prediction-hiv',
 'smol-property_prediction-sider',
 'presto-retrosynthesis',
 'presto-forward_reaction_prediction',
 'chebi-20-text2mol',
 'smol-property_prediction-esol',
 'alchemy_lumo',
 'qm9_homo_lumo_gap',
 'orderly-forward_reaction_prediction',
 'forward_reaction_prediction']